In [ ]:
"""
=============================================================================
  SPATIAL GEOSTATISTICAL ANALYSIS — Phytophthora Root Rot (PRR) in Avocado
=============================================================================
  Study area : Three commercial avocado lots, Antioquia, Colombia
  Data source : Field survey records — real plant-level severity evaluations

  All data read from the same unified workbook as the temporal analysis:
    PRR_Unified_data_ThreeLots.xlsx
      Sheet "Donmatias-L1"  →  903 plants
      Sheet "El Retiro-L2"  →  1 338 plants
      Sheet "La Ceja-L3"    →  1 666 plants

  Coordinate handling
  -------------------
    Northing / Easting columns are already in metres (UTM for Donmatias;
    projected coordinates for El Retiro and La Ceja).  Each lot's mean is
    subtracted for numerical stability in variogram and kriging computations.

  Analyses
  --------
    1. Indicator Kriging (IK)
         Variable   Z = 1 if severity ≥ 1, else 0  (incidence indicator)
         Variogram  Spherical / Exponential / Gaussian (NLS); sill ≤ p(1−p)
         Kriging    Ordinary Kriging on Z → P(disease presence) ∈ [0, 1]

    2. Universal Kriging (UK)
         Variable   √AUDPC per 180-day window
         Variogram  Spherical / Exponential / Gaussian (NLS)
         Kriging    Universal Kriging with linear regional drift
         Note       Large lots (>600 plants) use a representative subsample
                    (n = UK_SUBSAMPLE) for the UK system to remain tractable.

    3. Getis-Ord Gi* statistic
         Variable   Raw incidence (0–5) at 8 temporal snapshots
         Threshold  Median variogram range (biologically informed bandwidth)
         Output     Hot-spot / cold-spot classification at α = 0.05 / 0.01

  Outputs  (written to OUTPUT_DIR)
  ---------------------------------
    Fig_IK_Variograms.png / .pdf       Compact IK variograms — 3 lots × 7 periods
    Fig_UK_Variograms.png / .pdf       Compact UK variograms — 3 lots × 7 periods
    Fig_IK_UK_Maps_RetLac.png / .pdf   IK + UK interpolation maps (El Retiro & La Ceja)
    Fig_GiStar_RetLac.png / .pdf       Gi* hotspot maps (El Retiro & La Ceja)
    Fig_Donmatias_IK_UK.png / .pdf     Combined IK + UK figure for Donmatias
    Table_Geostatistical_Models.docx   Variogram model metrics (Word, landscape)

  Dependencies
  ------------
    pip install pandas numpy scipy matplotlib openpyxl pykrige python-docx

  Usage
  -----
    1. Place PRR_Unified_data_ThreeLots.xlsx in the same folder as this script.
    2. python spatial_analysis.py
=============================================================================
"""

import os
import warnings
import numpy  as np
import pandas as pd
from scipy.spatial.distance import cdist
from scipy.optimize         import curve_fit
from scipy.spatial          import ConvexHull, Delaunay
from scipy.ndimage          import gaussian_filter
from scipy.stats            import norm as sp_norm
from pykrige.ok import OrdinaryKriging
from pykrige.uk import UniversalKriging
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot    as plt
import matplotlib.gridspec  as gridspec
import matplotlib.ticker    as mticker
import matplotlib.patches   as mpatches
from matplotlib.colors import Normalize
from matplotlib.lines  import Line2D
warnings.filterwarnings("ignore")


# =============================================================================
# 0.  CONFIGURATION  ← only edit this block
# =============================================================================

UNIFIED_XLSX = "PRR_Unified_data_ThreeLots.xlsx"

SHEETS = {
    "Donmatias-L1" : "Donmatias-L1",
    "El Retiro-L2" : "El Retiro-L2",
    "La Ceja-L3"   : "La Ceja-L3",
}

OUTPUT_DIR   = "outputs"
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Seven 180-day evaluation windows
PERIOD_BOUNDS = [
    ("P1",    0,  180), ("P2",  180,  360), ("P3",  360,  540),
    ("P4",  540,  720), ("P5",  720,  900), ("P6",  900, 1080),
    ("P7", 1080, 1200),
]

INCIDENCE_THRESHOLD = 1      # severity ≥ 1 → Z = 1
N_LAGS              = 12     # lag classes in empirical variogram
PCT_MAXDIST         = 45     # percentile cap on pairwise distances
GRID_STEPS          = 60     # interpolation grid resolution (60 × 60)
UK_SUBSAMPLE        = 600    # max plants used for UK (O(n²) cost control)
GI_TIME_SHOW        = [0, 180, 360, 540, 720, 900, 1080, 1200]
DPI                 = 500
SMOOTH              = 0.5    # Gaussian smoothing σ for kriging map display


# =============================================================================
# 1.  DATA LOADING & PREPROCESSING
# =============================================================================

def load_lot(xlsx_path: str, sheet_name: str, label: str) -> pd.DataFrame:
    """
    Load one lot and centre coordinates for numerical stability.
    Coordinates are subtracted of their per-lot mean so that the origin
    is at the centroid of each lot; units remain metres.
    """
    if not os.path.exists(xlsx_path):
        raise RuntimeError(
            f"Workbook not found: '{xlsx_path}'\n"
            f"Working directory: {os.path.abspath('.')}")

    df = pd.read_excel(xlsx_path, sheet_name=sheet_name, skiprows=METADATA_ROWS)
    df.columns = [str(c).strip() for c in df.columns]
    df = df.dropna(how="all").reset_index(drop=True)

    # Centre coordinates per lot
    df["Northing"] -= df["Northing"].mean()
    df["Easting"]  -= df["Easting"].mean()

    print(f"  [{label}]  {len(df):>5} plants  |  "
          f"N [{df['Northing'].min():.0f}, {df['Northing'].max():.0f}] m  |  "
          f"E [{df['Easting'].min():.0f}, {df['Easting'].max():.0f}] m")
    return df


def load_all_lots() -> dict:
    print(f"\nReading workbook: {UNIFIED_XLSX}")
    dfs = {}
    for label, sheet in SHEETS.items():
        try:
            dfs[label] = load_lot(UNIFIED_XLSX, sheet, label)
        except Exception as exc:
            print(f"  [{label}]  ERROR: {exc}")
    return dfs


def compute_audpc_sqrt(df: pd.DataFrame) -> pd.DataFrame:
    """
    Add two columns per 180-day period:
      AUDPC_{Pk}       — trapezoidal AUDPC for each plant
      AUDPC_{Pk}_sqrt  — √AUDPC (variance-stabilising transformation)
    """
    dcols = sorted(
        [c for c in df.columns if c.startswith("D") and c[1:].isdigit()],
        key=lambda x: int(x[1:]))
    for pname, lo, hi in PERIOD_BOUNDS:
        cols = [c for c in dcols if lo <= int(c[1:]) <= hi]
        if len(cols) < 2: continue
        times = [int(c[1:]) for c in cols]
        vals  = df[cols].values
        df[f"AUDPC_{pname}"]      = np.trapezoid(vals, times, axis=1)
        df[f"AUDPC_{pname}_sqrt"] = np.sqrt(np.maximum(df[f"AUDPC_{pname}"], 0))
    return df


def compute_indicators(df: pd.DataFrame) -> tuple:
    """
    Add binary indicator column IND_{Pk} = 1 if any severity in the period
    reaches INCIDENCE_THRESHOLD, else 0.
    Returns (df, ind_cols_list).
    """
    dcols = sorted(
        [c for c in df.columns if c.startswith("D") and c[1:].isdigit()],
        key=lambda x: int(x[1:]))
    ind_cols = []
    for pname, lo, hi in PERIOD_BOUNDS:
        cols = [c for c in dcols if lo <= int(c[1:]) <= hi]
        if cols:
            df[f"IND_{pname}"] = (
                df[cols].max(axis=1) >= INCIDENCE_THRESHOLD
            ).astype(float)
            ind_cols.append(f"IND_{pname}")
    return df, ind_cols


# =============================================================================
# 2.  VARIOGRAM ANALYSIS
# =============================================================================

def empirical_variogram(coords: np.ndarray, vals: np.ndarray) -> tuple:
    """
    Method-of-moments estimator (Matheron 1963).
    Returns (lags, gammas) arrays with N_LAGS entries (or fewer if sparse).
    """
    dists    = cdist(coords, coords)
    v        = vals.astype(float)
    max_d    = np.percentile(dists[dists > 0], PCT_MAXDIST)
    lag_size = max_d / N_LAGS
    lags, gammas = [], []
    for i in range(N_LAGS):
        lo, hi = i * lag_size, (i+1) * lag_size
        mask   = (dists > lo) & (dists <= hi)
        pi, pj = np.where(mask)
        if len(pi) > 10:
            lags.append((lo+hi) / 2)
            gammas.append(((v[pi] - v[pj])**2).mean() / 2)
    return np.array(lags), np.array(gammas)


# Theoretical models
def _sph(h,n,s,r):  return np.where(h<=r, n+(s-n)*(1.5*(h/r)-0.5*(h/r)**3), s)
def _exp(h,n,s,r):  return n+(s-n)*(1-np.exp(-h/r))
def _gau(h,n,s,r):  return n+(s-n)*(1-np.exp(-(h/r)**2))

MODELS_FN = {"Spherical": _sph, "Exponential": _exp, "Gaussian": _gau}
PYKRIGE   = {"Spherical":"spherical","Exponential":"exponential","Gaussian":"gaussian"}


def fit_variogram(lags: np.ndarray, gammas: np.ndarray,
                  sill_upper: float = None) -> tuple:
    """
    Fit all three theoretical models by NLS.
    sill_upper : if given (IK), bounds the sill to p*(1-p).
    Returns (best_name, params, r2, all_results).
    """
    s0   = min(gammas.max(), sill_upper*0.9) if sill_upper else gammas.max()
    r0   = lags.max() * 0.5
    n0   = gammas.min() * 0.05
    ub_s = sill_upper * 1.5 if sill_upper else gammas.max() * 2
    best_name, best_p, best_r2 = None, None, -np.inf
    all_res = {}
    for name, func in MODELS_FN.items():
        try:
            popt, _ = curve_fit(func, lags, gammas,
                                p0=[n0, s0, r0],
                                bounds=([0, 0, 0], [ub_s, ub_s*2, lags.max()*2]),
                                maxfev=8000)
            pred   = func(lags, *popt)
            ss_res = np.sum((gammas - pred)**2)
            ss_tot = np.sum((gammas - gammas.mean())**2)
            r2     = 1 - ss_res/ss_tot if ss_tot > 0 else 0
            all_res[name] = dict(params=popt, r2=r2,
                                 rmse=np.sqrt(ss_res/len(lags)))
            if r2 > best_r2: best_r2=r2; best_name=name; best_p=popt
        except:
            all_res[name] = None
    return best_name, best_p, best_r2, all_res


def run_variogram_analysis(df: pd.DataFrame, cols: list,
                           label: str, is_ik: bool = False) -> tuple:
    """
    Fit variograms for all periods.
    Returns (vd list, metric_rows list).
    """
    coords = df[["Northing","Easting"]].values
    vd, rows = [], []
    for idx, col in enumerate(cols):
        period = idx + 1
        z      = df[col].fillna(0).values.astype(float)
        p_hat  = z.mean()
        lags, gammas = empirical_variogram(coords, z)
        if len(lags) < 3:
            vd.append(dict(period=period, pure_nugget=True, p_hat=p_hat))
            continue
        su = p_hat*(1-p_hat) if is_ik else None
        best_name, params, r2, all_res = fit_variogram(lags, gammas, su)
        if best_name is None:
            vd.append(dict(period=period, pure_nugget=True, p_hat=p_hat))
            continue
        nu, si, ra = params
        ci = (nu/si*100) if si > 0 else 0
        print(f"  [{label}] P{period}: {best_name:12s}  "
              f"R²={r2:.3f}  range={ra:.1f} m  CI={ci:.1f}%"
              + (f"  p(1-p)={p_hat*(1-p_hat):.4f}" if is_ik else ""))
        vd.append(dict(period=period, pure_nugget=False, lags=lags, gammas=gammas,
                       best_name=best_name, params=params, r2=r2,
                       all_results=all_res, p_hat=p_hat))
        for mn, mr in all_res.items():
            if mr is None: continue
            nu_m, si_m, ra_m = mr["params"]
            ci_m = (nu_m/si_m*100) if si_m > 0 else 0
            entry = dict(Lot=label, Period=f"P{period}", Model=mn,
                         Nugget=round(nu_m,4), Sill=round(si_m,4),
                         Range_m=round(ra_m,1), R2=round(mr["r2"],4),
                         RMSE=round(mr["rmse"],4), CI_pct=round(ci_m,1),
                         Best="*" if mn==best_name else "")
            if is_ik: entry["p_hat"] = round(p_hat, 4)
            rows.append(entry)
    return vd, rows


# =============================================================================
# 3.  KRIGING
# =============================================================================

def _make_grid(df: pd.DataFrame) -> tuple:
    """Build normalised grid for kriging; returns coordinates and scale info."""
    x = df["Easting"].values.astype(float)
    y = df["Northing"].values.astype(float)
    xmin, xmax = x.min(), x.max()
    ymin, ymax = y.min(), y.max()
    span = max(xmax-xmin, ymax-ymin)
    xn   = (x-xmin)/span;  yn = (y-ymin)/span
    xi   = np.linspace(0, (xmax-xmin)/span, GRID_STEPS)
    yi   = np.linspace(0, (ymax-ymin)/span, GRID_STEPS)
    return x, y, xn, yn, xi, yi, xmin, xmax, ymin, ymax, span


def _idw(xn, yn, xi, yi, z) -> np.ndarray:
    """IDW fallback when kriging fails."""
    XX, YY = np.meshgrid(xi, yi)
    g = np.zeros_like(XX, dtype=float)
    for i in range(GRID_STEPS):
        for j in range(GRID_STEPS):
            d2 = np.maximum((xn-XX[i,j])**2 + (yn-YY[i,j])**2, 1e-10)
            g[i,j] = np.sum(z/d2) / np.sum(1/d2)
    return np.clip(g, 0, None)


def run_indicator_kriging(df: pd.DataFrame, vd: list, ind_cols: list) -> list:
    """
    Ordinary Kriging on the binary indicator Z ∈ {0,1}.
    Output: P(disease presence) ∈ [0, 1] on a regular grid.
    Falls back to IDW if kriging fails.
    """
    x, y, xn, yn, xi, yi, xmin, xmax, ymin, ymax, span = _make_grid(df)
    vdmap   = {v["period"]: v for v in vd}
    results = []
    for idx, col in enumerate(ind_cols):
        period = idx+1
        z      = df[col].fillna(0).values.astype(float)
        vdr    = vdmap.get(period, {})
        pg     = None
        if not vdr.get("pure_nugget") and vdr.get("best_name"):
            nu, si, ra = vdr["params"]
            try:
                OK = OrdinaryKriging(
                    xn, yn, z,
                    variogram_model=PYKRIGE[vdr["best_name"]],
                    variogram_parameters=[max(si-nu,1e-6), max(ra/span,1e-5), max(nu,0)],
                    verbose=False, enable_plotting=False)
                raw, _ = OK.execute("grid", xi, yi)
                pg = np.clip(np.array(raw), 0.0, 1.0)
            except: pass
        if pg is None:
            pg = np.clip(_idw(xn, yn, xi, yi, z), 0, 1)
        results.append(dict(
            period=period, z_grid=pg, p_hat=z.mean(),
            xi=xi*(xmax-xmin)+xmin, yi=yi*(ymax-ymin)+ymin,
            x_obs=x, y_obs=y, z_obs=z))
        print(f"  IK P{period}: p̂={z.mean():.3f}  "
              f"grid=[{pg.min():.3f}, {pg.max():.3f}]")
    return results


def run_universal_kriging(df: pd.DataFrame, vd: list,
                           audpc_cols: list, rng=None) -> list:
    """
    Universal Kriging with linear regional drift applied to √AUDPC.
    For lots with > UK_SUBSAMPLE plants a random subsample is used
    to keep the kriging system tractable.
    Falls back to IDW if kriging fails.
    """
    if rng is None: rng = np.random.default_rng(42)
    x, y, xn, yn, xi, yi, xmin, xmax, ymin, ymax, span = _make_grid(df)
    vdmap   = {v["period"]: v for v in vd}
    results = []
    for idx, col in enumerate(audpc_cols):
        period = idx+1
        z      = df[col].fillna(0).values.astype(float)
        vdr    = vdmap.get(period, {})
        zg     = None
        if not vdr.get("pure_nugget") and vdr.get("best_name"):
            nu, si, ra = vdr["params"]
            if len(xn) > UK_SUBSAMPLE:
                ii = rng.choice(len(xn), UK_SUBSAMPLE, replace=False)
                xs, ys, zs = xn[ii], yn[ii], z[ii]
            else:
                xs, ys, zs = xn, yn, z
            try:
                UK = UniversalKriging(
                    xs, ys, zs,
                    variogram_model=PYKRIGE[vdr["best_name"]],
                    variogram_parameters=[max(si-nu,0.01), max(ra/span,1e-4), max(nu,0)],
                    drift_terms=["regional_linear"],
                    verbose=False, enable_plotting=False)
                raw, _ = UK.execute("grid", xi, yi)
                zg = np.clip(np.array(raw), 0, z.max()*1.5)
            except: pass
        if zg is None:
            zg = _idw(xn, yn, xi, yi, z)
        results.append(dict(
            period=period, z_grid=zg,
            xi=xi*(xmax-xmin)+xmin, yi=yi*(ymax-ymin)+ymin,
            x_obs=x, y_obs=y, z_obs=z))
        print(f"  UK P{period}: grid=[{zg.min():.2f}, {zg.max():.2f}]")
    return results


# =============================================================================
# 4.  GETIS-ORD Gi* STATISTIC
# =============================================================================

def gi_star(coords: np.ndarray, values: np.ndarray, d: float) -> tuple:
    """
    Getis-Ord Gi* z-score for each observation.
    Binary spatial weight matrix W: wij = 1 if dist ≤ d (self-included).
    Reference: Ord & Getis (1995) Geographical Analysis 27:286–306.
    """
    n     = len(values)
    x     = values.astype(float)
    xbar  = x.mean()
    s     = x.std(ddof=0)
    dists = cdist(coords, coords)
    W     = (dists <= d).astype(float)
    wi    = W.sum(axis=1)
    wi2   = (W**2).sum(axis=1)
    wx    = W @ x
    num   = wx - xbar * wi
    den   = s * np.sqrt(np.maximum((n*wi2 - wi**2) / (n-1), 1e-10))
    z     = np.where(den > 0, num/den, 0.0)
    p     = 2 * (1 - sp_norm.cdf(np.abs(z)))
    return z, p


def classify_gi(z: np.ndarray, p: np.ndarray) -> np.ndarray:
    cats = np.full(len(z), "Not Significant", dtype=object)
    cats[(z >  1.960) & (p < 0.05)] = "Hot Spot (95%)"
    cats[(z >  2.576) & (p < 0.01)] = "Hot Spot (99%)"
    cats[(z < -1.960) & (p < 0.05)] = "Cold Spot (95%)"
    cats[(z < -2.576) & (p < 0.01)] = "Cold Spot (99%)"
    return cats


# =============================================================================
# 5.  GLOBAL PLOT STYLE
# =============================================================================

BLACK  = "#000000"
LGRAY  = "#CCCCCC"
DGRAY  = "#555555"
LOT_CLR = {
    "Donmatias-L1": "#1B4F8A",
    "El Retiro-L2": "#C0392B",
    "La Ceja-L3":   "#1E8449",
}
LOT_LBL = {"Donmatias-L1":"A","El Retiro-L2":"B","La Ceja-L3":"C"}
PERIOD_LABELS = [f"D{(p-1)*180}–D{min(p*180,1200)}" for p in range(1,8)]
CMAP_IK = plt.cm.viridis
CMAP_UK = plt.cm.plasma

CAT_STYLE = {
    "Hot Spot (99%)":  dict(color="#B22222",s=5,alpha=0.90,zorder=5),
    "Hot Spot (95%)":  dict(color="#FF7B7B",s=4,alpha=0.80,zorder=4),
    "Not Significant": dict(color="#C8C8C8",s=2.5,alpha=0.50,zorder=2),
    "Cold Spot (95%)": dict(color="#5BA4D4",s=4,alpha=0.80,zorder=4),
    "Cold Spot (99%)": dict(color="#08519C",s=5,alpha=0.90,zorder=5),
}
DRAW_ORDER = ["Not Significant","Cold Spot (95%)","Cold Spot (99%)",
              "Hot Spot (95%)","Hot Spot (99%)"]

plt.rcParams.update({
    "font.family":      "sans-serif",
    "font.sans-serif":  ["DejaVu Sans","Helvetica"],
    "font.size":        11,
    "axes.linewidth":   0.8,
    "figure.facecolor":"white",
    "axes.facecolor":  "#FAFAFA",
    "axes.grid":        False,
    "xtick.major.size": 3.5, "ytick.major.size": 3.5,
    "xtick.major.width":0.8, "ytick.major.width":0.8,
    "xtick.direction":  "out","ytick.direction": "out",
    "pdf.fonttype":     42,
})

LEGEND_HANDLES_VARIO = [
    Line2D([0],[0], color=BLACK,  lw=2.0, label="Fitted curve"),
    Line2D([0],[0], marker="^", color="w", lw=0, ms=8,
           markerfacecolor=DGRAY, label="Adjusted values"),
    Line2D([0],[0], marker="o", color="w", lw=0, ms=8,
           markeredgecolor=BLACK, markeredgewidth=1.3, label="Empirical values"),
    Line2D([0],[0], color=LGRAY, lw=0.8, ls="--", label="Sill / Range"),
]


def hull_mask(xi, yi, xo, yo) -> np.ndarray:
    """Boolean mask: True inside the convex hull of observed points."""
    pts = np.column_stack([xo, yo])
    try:
        hull = ConvexHull(pts)
        tri  = Delaunay(pts[hull.vertices])
        XI, YI = np.meshgrid(xi, yi)
        inside = tri.find_simplex(
            np.column_stack([XI.ravel(), YI.ravel()])) >= 0
        return inside.reshape(XI.shape)
    except:
        return np.ones((len(yi), len(xi)), dtype=bool)


def _save(fig, name: str) -> None:
    for ext in ("png", "pdf"):
        fig.savefig(os.path.join(OUTPUT_DIR, f"{name}.{ext}"),
                    dpi=DPI if ext == "png" else None,
                    bbox_inches="tight", facecolor="white")
    plt.close(fig)
    print(f"  Saved: {name}")


# =============================================================================
# 6.  FIGURE: COMPACT VARIOGRAMS  (3 lots × 7 periods, 2×4 block per lot)
# =============================================================================

def _draw_vario_panel(ax, v, clr, show_x: bool, show_y: bool) -> None:
    ax.set_facecolor("#FAFAFA")
    for sp in ax.spines.values(): sp.set_color(LGRAY); sp.set_linewidth(0.7)
    if not (v and not v.get("pure_nugget") and v.get("best_name")):
        ax.text(0.5, 0.5, "Pure\nnugget", transform=ax.transAxes,
                ha="center", va="center", fontsize=10, color=DGRAY,
                bbox=dict(boxstyle="round,pad=0.3",fc="white",ec=LGRAY,lw=0.5))
        ax.set_xticks([]); ax.set_yticks([]); return

    lags=v["lags"]; gammas=v["gammas"]; best=v["best_name"]
    params=v["params"]; nu,si,ra=params
    r2=v["all_results"][best]["r2"]; func=MODELS_FN[best]

    ax.axhline(si, color=LGRAY, lw=0.8, ls="--", zorder=1)
    ax.axvline(ra, color=LGRAY, lw=0.7, ls="--", zorder=1)

    h = np.linspace(0, lags.max()*1.07, 600)
    ax.plot(h, func(h,*params), color=clr, lw=2.2, zorder=3)
    ax.scatter(lags, func(lags,*params), marker="^", s=36,
               color=clr, zorder=4, linewidths=0, alpha=0.9)
    ax.scatter(lags, gammas, s=40, color="white",
               edgecolors=clr, linewidths=1.4, zorder=5)
    ax.plot(0, nu, "o", ms=5.5, color="white",
            markeredgecolor=clr, markeredgewidth=1.4, zorder=6, clip_on=False)

    ax.set_xlim(left=0, right=lags.max()*1.08)
    ax.set_ylim(bottom=0, top=max(gammas.max(), si)*1.18)
    ax.yaxis.set_major_locator(mticker.MaxNLocator(4))
    ax.xaxis.set_major_locator(mticker.MaxNLocator(4, integer=True))
    ax.tick_params(labelsize=10, pad=2, colors=BLACK)

    annot = (f"Model = {best}\n"
             f"Nugget = {nu:.3f}\n"
             f"Sill    = {si:.3f}\n"
             f"Range = {ra:.0f} m\n"
             f"R²     = {r2:.3f}")
    ax.text(0.97, 0.04, annot, transform=ax.transAxes,
            fontsize=9.5, va="bottom", ha="right",
            color=BLACK, fontfamily="monospace", linespacing=1.45,
            bbox=dict(boxstyle="round,pad=0.38", fc="white",
                      ec=LGRAY, lw=0.6, alpha=0.96))
    if show_x: ax.set_xlabel("Distance (m)", fontsize=11, labelpad=3, color=BLACK)
    else: ax.tick_params(axis="x", labelbottom=False)
    if show_y: ax.set_ylabel("Semivariance", fontsize=11, labelpad=3, color=BLACK)
    else: ax.tick_params(axis="y", labelleft=False)


def figure_compact_variograms(vario_dict: dict, suptitle: str, fname: str) -> None:
    """
    Three lot-blocks stacked vertically.
    Each block: 2 rows × 4 cols (periods 1–4 top, 5–7 + legend bottom).
    """
    lots = [l for l in ["Donmatias-L1","El Retiro-L2","La Ceja-L3"]
            if l in vario_dict]
    fig  = plt.figure(figsize=(18, 22), facecolor="white")
    outer = gridspec.GridSpec(3, 1, figure=fig,
                               hspace=0.28,
                               left=0.07, right=0.99,
                               top=0.96,  bottom=0.04)
    for bi, lot in enumerate(lots):
        clr   = LOT_CLR[lot]
        vd    = vario_dict[lot]
        vdmap = {v["period"]: v for v in vd}
        inner = gridspec.GridSpecFromSubplotSpec(2, 4,
                    subplot_spec=outer[bi], hspace=0.24, wspace=0.16)
        # Section label
        ax_d = fig.add_subplot(outer[bi]); ax_d.set_visible(False)
        pos  = ax_d.get_position()
        fig.text(0.005, pos.y1+0.004,
                 LOT_LBL[lot], fontsize=20, fontweight="bold",
                 color=BLACK, va="bottom")
        fig.text(0.025, pos.y1+0.004,
                 lot, fontsize=13, fontweight="bold",
                 color=clr, va="bottom")
        # Panels
        for idx in range(7):
            row, col = divmod(idx, 4)
            ax = fig.add_subplot(inner[row, col])
            _draw_vario_panel(ax, vdmap.get(idx+1), clr,
                              show_x=(row==1), show_y=(col==0))
            ax.set_title(f"Period {idx+1}  ({PERIOD_LABELS[idx]})",
                         fontsize=11, fontweight="bold", pad=5, color=BLACK)
        # Legend
        ax_leg = fig.add_subplot(inner[1, 3]); ax_leg.set_visible(False)
        pos_l  = ax_leg.get_position()
        leg = fig.legend(handles=LEGEND_HANDLES_VARIO,
                         fontsize=10.5, loc="center",
                         bbox_to_anchor=(pos_l.x0+pos_l.width/2,
                                         pos_l.y0+pos_l.height/2),
                         bbox_transform=fig.transFigure,
                         framealpha=0.95, edgecolor=LGRAY, fancybox=False,
                         borderpad=0.8, handlelength=1.8,
                         handletextpad=0.5, labelspacing=0.5,
                         title="Legend", title_fontsize=11)
        leg.get_title().set_fontweight("bold")
        leg.get_title().set_color(BLACK)
    fig.suptitle(suptitle, fontsize=14, fontweight="bold", y=0.99, color=BLACK)
    _save(fig, fname)


# =============================================================================
# 7.  FIGURE: IK + UK MAPS  (El Retiro & La Ceja)
# =============================================================================

def _draw_map_panel(ax, kdat, cmap, norm, show_x, show_y,
                    p_label, badge=None, row_lbl=None) -> None:
    xi, yi = kdat["xi"], kdat["yi"]
    zg     = np.clip(gaussian_filter(kdat["z_grid"], sigma=SMOOTH), 0, None)
    xo, yo, zo = kdat["x_obs"], kdat["y_obs"], kdat["z_obs"]
    mask   = hull_mask(xi, yi, xo, yo)
    zg_m   = np.where(mask, zg, np.nan)
    ax.imshow(zg_m, origin="lower",
              extent=[xi.min(),xi.max(),yi.min(),yi.max()],
              cmap=cmap, norm=norm, aspect="auto",
              interpolation="bilinear", zorder=2)
    ax.set_axisbelow(True)
    ax.grid(True, color="#EEEEEE", lw=0.4, zorder=1)
    ax.scatter(xo, yo, c=zo, cmap=cmap, norm=norm,
               s=0.8, linewidths=0, zorder=3, alpha=0.30)
    ax.set_title(p_label, fontsize=9.5, fontweight="bold", pad=3, color=BLACK)
    if badge:
        ax.text(0.97, 0.97, badge, transform=ax.transAxes,
                fontsize=8.5, color=BLACK, ha="right", va="top",
                bbox=dict(boxstyle="round,pad=0.22", fc="white",
                          ec="none", alpha=0.82))
    ax.tick_params(labelsize=8.5, colors=BLACK, pad=2)
    ax.xaxis.set_major_locator(mticker.MaxNLocator(3, integer=True))
    ax.yaxis.set_major_locator(mticker.MaxNLocator(4, integer=True))
    ax.set_facecolor("white")
    for sp in ax.spines.values(): sp.set_color(LGRAY); sp.set_linewidth(0.4)
    if show_x: ax.set_xlabel("Easting (m)", fontsize=9.5, labelpad=3)
    else: ax.tick_params(axis="x", labelbottom=False)
    if show_y and row_lbl:
        ax.set_ylabel(row_lbl, fontsize=7.5, labelpad=3,
                      fontweight="bold", color=BLACK)
    elif not show_y:
        ax.tick_params(axis="y", labelleft=False)


def figure_ik_uk_maps(ik_kr_dict, uk_kr_dict, lots, fname) -> None:
    """4 rows (IK-L1, IK-L2, UK-L1, UK-L2) × 7 cols with shared colorbars."""
    from matplotlib.cm import ScalarMappable
    IK_NORM = Normalize(0, 1)
    uk_vmax = max(
        np.percentile([k["z_grid"].max() for k in uk_kr_dict[l]], 95)
        for l in lots)
    UK_NORM = Normalize(0, uk_vmax)
    row_cfg = [
        (lots[0],"IK",ik_kr_dict[lots[0]],CMAP_IK,IK_NORM,True),
        (lots[1],"IK",ik_kr_dict[lots[1]],CMAP_IK,IK_NORM,False),
        (lots[0],"UK",uk_kr_dict[lots[0]],CMAP_UK,UK_NORM,True),
        (lots[1],"UK",uk_kr_dict[lots[1]],CMAP_UK,UK_NORM,False),
    ]
    fig = plt.figure(figsize=(20, 13), facecolor="white")
    gs  = gridspec.GridSpec(4, 8, figure=fig,
                             hspace=0.12, wspace=0.06,
                             left=0.07, right=0.96,
                             top=0.93,  bottom=0.06,
                             width_ratios=[1]*7+[0.06])
    for ri, (lot, method, kr_list, cmap, norm, show_y) in enumerate(row_cfg):
        row_lbl = f"{lot}\n({method})"
        for ci, kdat in enumerate(kr_list):
            ax = fig.add_subplot(gs[ri, ci])
            badge = (f"p̂={kdat['p_hat']:.1%}"
                     if method=="IK" and "p_hat" in kdat else None)
            _draw_map_panel(ax, kdat, cmap, norm,
                            show_x=(ri==3), show_y=(ci==0 and show_y),
                            p_label=PERIOD_LABELS[ci], badge=badge,
                            row_lbl=row_lbl if ci==0 else None)
            if ri==0:
                ax.set_title(PERIOD_LABELS[ci], fontsize=8,
                             fontweight="bold", pad=4, color=BLACK)
    cax_ik = fig.add_subplot(gs[0:2, 7])
    cb_ik  = plt.colorbar(ScalarMappable(norm=IK_NORM, cmap=CMAP_IK), cax=cax_ik)
    cb_ik.set_label("P(presence)", fontsize=8, labelpad=3)
    cb_ik.ax.tick_params(labelsize=7.5); cb_ik.outline.set_linewidth(0.4)
    cax_uk = fig.add_subplot(gs[2:4, 7])
    cb_uk  = plt.colorbar(ScalarMappable(norm=UK_NORM, cmap=CMAP_UK), cax=cax_uk)
    cb_uk.set_label("√AUDPC", fontsize=8, labelpad=3)
    cb_uk.ax.tick_params(labelsize=7.5); cb_uk.outline.set_linewidth(0.4)
    fig.suptitle(
        f"Spatial interpolation of PRR — "
        f"Indicator Kriging (rows 1–2) and Universal Kriging (rows 3–4)\n"
        f"{lots[0]} and {lots[1]}",
        fontsize=11, fontweight="bold", y=0.98, color=BLACK)
    _save(fig, fname)


# =============================================================================
# 8.  FIGURE: Gi* HOTSPOT MAPS  (El Retiro & La Ceja)
# =============================================================================

def figure_gistar(lot_configs: list, fname: str) -> None:
    """2 rows × 8 time snapshots."""
    fig = plt.figure(figsize=(20, 9), facecolor="white")
    gs  = gridspec.GridSpec(2, 8, figure=fig,
                             hspace=0.10, wspace=0.08,
                             left=0.06, right=0.97,
                             top=0.91,  bottom=0.09)
    for ri, (lot, df, d_thresh, lot_clr) in enumerate(lot_configs):
        coords = df[["Easting","Northing"]].values
        for ci, t in enumerate(GI_TIME_SHOW):
            ax  = fig.add_subplot(gs[ri, ci])
            col = f"D{t}"
            if col not in df.columns: ax.set_visible(False); continue
            sev  = df[col].fillna(0).values.astype(float)
            z, p = gi_star(coords, sev, d_thresh)
            cats = classify_gi(z, p)
            x, y = df["Easting"].values, df["Northing"].values
            for cat in DRAW_ORDER:
                m = cats == cat
                if not m.any(): continue
                st = CAT_STYLE[cat]
                ax.scatter(x[m], y[m], c=st["color"], s=st["s"],
                           alpha=st["alpha"], linewidths=0,
                           zorder=st["zorder"], rasterized=True)
            ax.set_facecolor("white"); ax.set_axisbelow(True)
            ax.grid(True, color="#F0F0F0", lw=0.4, zorder=1)
            for sp in ax.spines.values():
                sp.set_color(LGRAY); sp.set_linewidth(0.4)
            ax.tick_params(labelsize=6, pad=1.5, colors=BLACK)
            ax.xaxis.set_major_locator(mticker.MaxNLocator(3, integer=True))
            ax.yaxis.set_major_locator(mticker.MaxNLocator(3, integer=True))
            if ri==0: ax.set_title(f"{t} days", fontsize=8,
                                    fontweight="bold", pad=3, color=BLACK)
            if ri==1: ax.set_xlabel("Easting (m)", fontsize=7, labelpad=2)
            else: ax.tick_params(axis="x", labelbottom=False)
            if ci==0: ax.set_ylabel(f"{lot}\nNorthing (m)", fontsize=7.5,
                                     labelpad=2, color=lot_clr, fontweight="bold")
            else: ax.tick_params(axis="y", labelleft=False)
            ax.set_aspect("equal", adjustable="datalim")
    handles = [
        Line2D([0],[0], marker="o", color="w", markersize=6,
               markerfacecolor=CAT_STYLE[c]["color"], label=c)
        for c in ["Hot Spot (99%)","Hot Spot (95%)","Not Significant",
                   "Cold Spot (95%)","Cold Spot (99%)"]
    ]
    fig.legend(handles=handles, ncol=5, fontsize=8,
               loc="lower center", bbox_to_anchor=(0.5, 0.01),
               framealpha=0.95, edgecolor=LGRAY, fancybox=False,
               borderpad=0.5, handlelength=1.2,
               handletextpad=0.4, columnspacing=0.8)
    fig.suptitle("Getis-Ord Gi* spatial clustering of PRR severity",
                 fontsize=11, fontweight="bold", y=0.98, color=BLACK)
    _save(fig, fname)


# =============================================================================
# 9.  FIGURE: DONMATIAS IK + UK SIDE-BY-SIDE
# =============================================================================

def figure_donmatias_ik_uk(ik_kr: list, uk_kr: list, fname: str) -> None:
    """
    3 rows (P1–P3 / P4–P6 / P7+legends), IK on the left | UK on the right.
    No vertical separator line.
    """
    IK_NORM = Normalize(0, 1)
    UK_VMAX = float(np.percentile([k["z_grid"].max() for k in uk_kr], 95))
    UK_NORM = Normalize(0, UK_VMAX)

    FS_TITLE  = 15;  FS_BADGE  = 12;  FS_TICK   = 11.5
    FS_AX     = 12.5; FS_HTITLE = 14.5
    FS_LEG    = 12;  FS_LEG_T  = 13

    fig = plt.figure(figsize=(22, 22), facecolor="white")
    # 3 IK cols | wide gap | 3 UK cols
    COL_RATIOS = [1, 1, 1, 0.18, 1, 1, 1]
    outer = gridspec.GridSpec(3, 1, figure=fig,
                               hspace=0.12,
                               left=0.06, right=0.985,
                               top=0.940, bottom=0.048)

    def make_inner(spec):
        return gridspec.GridSpecFromSubplotSpec(
            1, 7, subplot_spec=spec,
            wspace=0.07, width_ratios=COL_RATIOS)

    def draw(ax, kdat, cmap, norm, show_x, show_y, badge, plbl):
        xi, yi = kdat["xi"], kdat["yi"]
        zg = np.clip(gaussian_filter(kdat["z_grid"], sigma=SMOOTH), 0, None)
        xo, yo, zo = kdat["x_obs"], kdat["y_obs"], kdat["z_obs"]
        mask = hull_mask(xi, yi, xo, yo)
        ax.imshow(np.where(mask,zg,np.nan),
                  origin="lower",
                  extent=[xi.min(),xi.max(),yi.min(),yi.max()],
                  cmap=cmap, norm=norm, aspect="auto",
                  interpolation="bilinear", zorder=2)
        ax.set_axisbelow(True)
        ax.grid(True, color="#EEEEEE", lw=0.45, zorder=1)
        ax.scatter(xo, yo, c=zo, cmap=cmap, norm=norm,
                   s=0.8, linewidths=0, zorder=3, alpha=0.30)
        ax.set_title(plbl, fontsize=FS_TITLE, fontweight="bold",
                     pad=4, color=BLACK)
        if badge:
            ax.text(0.97, 0.97, badge, transform=ax.transAxes,
                    fontsize=FS_BADGE, color=BLACK, ha="right", va="top",
                    bbox=dict(boxstyle="round,pad=0.28",
                              fc="white", ec="none", alpha=0.82))
        ax.tick_params(labelsize=FS_TICK, colors=BLACK, pad=2.5)
        ax.xaxis.set_major_locator(mticker.MaxNLocator(3, integer=True))
        ax.yaxis.set_major_locator(mticker.MaxNLocator(4, integer=True))
        ax.set_facecolor("white")
        for sp in ax.spines.values():
            sp.set_color(LGRAY); sp.set_linewidth(0.5)
        if show_x: ax.set_xlabel("Spatial dimension in X", fontsize=FS_AX, labelpad=4)
        else: ax.tick_params(axis="x", labelbottom=False)
        if show_y: ax.set_ylabel("Spatial dimension in Y", fontsize=FS_AX, labelpad=4)
        else: ax.tick_params(axis="y", labelleft=False)

    # Rows 0 and 1: P1–P3, P4–P6
    for row_i, pi_list in enumerate([[0,1,2],[3,4,5]]):
        g = make_inner(outer[row_i])
        for ci, pi in enumerate(pi_list):
            ax = fig.add_subplot(g[0, ci])
            draw(ax, ik_kr[pi], CMAP_IK, IK_NORM, False, (ci==0),
                 f"p̂={ik_kr[pi]['p_hat']:.1%}", PERIOD_LABELS[pi])
            ax = fig.add_subplot(g[0, ci+4])
            draw(ax, uk_kr[pi], CMAP_UK, UK_NORM, False, False,
                 None, PERIOD_LABELS[pi])
        fig.add_subplot(g[0, 3]).set_visible(False)

    # Row 2: P7 + legends
    g2 = make_inner(outer[2])
    ax = fig.add_subplot(g2[0, 0])
    draw(ax, ik_kr[6], CMAP_IK, IK_NORM, True, True,
         f"p̂={ik_kr[6]['p_hat']:.1%}", PERIOD_LABELS[6])

    ax_l1 = fig.add_subplot(g2[0, 1:3]); ax_l1.set_visible(False)
    pos = ax_l1.get_position()
    ik_patches = [mpatches.Patch(facecolor=CMAP_IK(i/4),
                                  edgecolor="#888", lw=0.5,
                                  label=f"{i*0.2:.2f} – {(i+1)*0.2:.2f}")
                  for i in range(4, -1, -1)]
    l1 = fig.legend(handles=ik_patches,
                    title="P(disease presence)", loc="center",
                    bbox_to_anchor=(pos.x0+pos.width/2, pos.y0+pos.height/2),
                    bbox_transform=fig.transFigure,
                    fontsize=FS_LEG, title_fontsize=FS_LEG_T,
                    framealpha=0.96, edgecolor=LGRAY, fancybox=False,
                    borderpad=0.85, handlelength=1.6,
                    handletextpad=0.6, labelspacing=0.55)
    l1.get_title().set_fontweight("bold")

    fig.add_subplot(g2[0, 3]).set_visible(False)

    ax = fig.add_subplot(g2[0, 4])
    draw(ax, uk_kr[6], CMAP_UK, UK_NORM, True, False, None, PERIOD_LABELS[6])

    ax_l2 = fig.add_subplot(g2[0, 5:7]); ax_l2.set_visible(False)
    pos = ax_l2.get_position()
    uk_b = np.linspace(0, UK_VMAX, 6)
    uk_patches = [mpatches.Patch(facecolor=CMAP_UK(i/4),
                                  edgecolor="#888", lw=0.5,
                                  label=f"{uk_b[i]:.1f} – {uk_b[i+1]:.1f}")
                  for i in range(4, -1, -1)]
    l2 = fig.legend(handles=uk_patches,
                    title="√AUDPC", loc="center",
                    bbox_to_anchor=(pos.x0+pos.width/2, pos.y0+pos.height/2),
                    bbox_transform=fig.transFigure,
                    fontsize=FS_LEG, title_fontsize=FS_LEG_T,
                    framealpha=0.96, edgecolor=LGRAY, fancybox=False,
                    borderpad=0.85, handlelength=1.6,
                    handletextpad=0.6, labelspacing=0.55)
    l2.get_title().set_fontweight("bold")

    # Half-titles — no vertical separator line
    fig.text(0.245, 0.962,
             "Indicator Kriging of PRR incidence P(presence) — Donmatias-L1",
             ha="center", va="bottom", fontsize=FS_HTITLE,
             fontweight="bold", color=BLACK)
    fig.text(0.755, 0.962,
             "Universal Kriging of PRR severity (√AUDPC) — Donmatias-L1",
             ha="center", va="bottom", fontsize=FS_HTITLE,
             fontweight="bold", color=BLACK)
    _save(fig, fname)


# =============================================================================
# 10.  WORD TABLE — GEOSTATISTICAL MODEL METRICS
# =============================================================================

def make_geostats_word_table(ik_rows: list, uk_rows: list) -> None:
    """
    Two-section landscape Word table:
      Section I  — IK variogram metrics
      Section II — UK variogram metrics
    Colour-coded by lot; best model marked with ✓.
    """
    try:
        from docx import Document
        from docx.shared    import Pt, RGBColor, Inches
        from docx.enum.text import WD_ALIGN_PARAGRAPH
        from docx.oxml.ns   import qn
        from docx.oxml      import OxmlElement
    except ImportError:
        print("  python-docx not installed — skipping Word table."); return

    LC = {
        "Donmatias-L1": {"row":"D6E4F0","best":"AED6F1","hdr":"1B4F8A"},
        "El Retiro-L2": {"row":"FADBD8","best":"F1948A","hdr":"C0392B"},
        "La Ceja-L3":   {"row":"D5F5E3","best":"ABEBC6","hdr":"1E8449"},
    }

    def rgb(h): return RGBColor(int(h[:2],16),int(h[2:4],16),int(h[4:],16))
    def bg(cell, hc):
        tc=cell._tc; p=tc.get_or_add_tcPr()
        s=OxmlElement("w:shd"); s.set(qn("w:val"),"clear")
        s.set(qn("w:color"),"auto"); s.set(qn("w:fill"),hc); p.append(s)
    def brd(cell, color="FFFFFF", sz=4):
        tc=cell._tc; p=tc.get_or_add_tcPr(); b=OxmlElement("w:tcBorders")
        for e in ["top","left","bottom","right"]:
            el=OxmlElement(f"w:{e}"); el.set(qn("w:val"),"single")
            el.set(qn("w:sz"),str(sz)); el.set(qn("w:color"),color); b.append(el)
        p.append(b)
    def ct(cell, text, bold=False, italic=False, sz=8.5,
           align=WD_ALIGN_PARAGRAPH.CENTER, color="1C2833"):
        pp=cell.paragraphs[0]; pp.alignment=align
        pp.paragraph_format.space_before=Pt(1)
        pp.paragraph_format.space_after =Pt(1)
        r=pp.add_run(str(text)); r.bold=bold; r.italic=italic
        r.font.size=Pt(sz); r.font.color.rgb=rgb(color)

    def add_section(doc, rows, col_hdrs, title, title_color):
        p=doc.add_paragraph(); p.alignment=WD_ALIGN_PARAGRAPH.CENTER
        r=p.add_run(title); r.bold=True; r.font.size=Pt(11)
        r.font.color.rgb=rgb(title_color)
        tbl=doc.add_table(rows=1,cols=len(col_hdrs)); tbl.style="Table Grid"
        for ci,lbl in enumerate(col_hdrs):
            c=tbl.rows[0].cells[ci]; bg(c,"1C2833"); brd(c)
            ct(c, lbl, bold=True, sz=9, color="FFFFFF")
        cur_lot=None; cur_period=None
        for row in rows:
            lc=LC.get(row.get("Lot",""),{"row":"F5F5F5","best":"DDDDDD","hdr":"555555"})
            is_best=row.get("Best","")=="*"
            if row.get("Lot")!=cur_lot:
                sep=tbl.add_row(); mc=sep.cells[0].merge(sep.cells[-1])
                bg(mc,lc["hdr"]); brd(mc,lc["hdr"],6)
                ct(mc,row["Lot"],bold=True,sz=9.5,
                   align=WD_ALIGN_PARAGRAPH.LEFT,color="FFFFFF")
                cur_lot=row["Lot"]; cur_period=None
            if row.get("Period")!=cur_period:
                per=tbl.add_row(); mc2=per.cells[0].merge(per.cells[-1])
                bg(mc2,"F2F2F2"); brd(mc2,"AAAAAA",3)
                ct(mc2,row["Period"],bold=True,italic=True,sz=9,
                   align=WD_ALIGN_PARAGRAPH.LEFT,color="444444")
                cur_period=row["Period"]
            dr=tbl.add_row(); fill=lc["best"] if is_best else lc["row"]
            for ci,key in enumerate(col_hdrs[:-1]):
                c=dr.cells[ci]; bg(c,fill); brd(c)
                align=(WD_ALIGN_PARAGRAPH.LEFT if ci<=1
                        else WD_ALIGN_PARAGRAPH.CENTER)
                ct(c, row.get(key,""), bold=is_best, sz=8.5, align=align)
            c=dr.cells[-1]; bg(c,fill); brd(c)
            ct(c, "✓" if is_best else "", bold=is_best, sz=8.5)
        doc.add_paragraph()

    doc=Document(); sec=doc.sections[0]
    sec.page_width=Inches(13); sec.page_height=Inches(8.5)
    for attr in ["top_margin","bottom_margin","left_margin","right_margin"]:
        setattr(sec, attr, Inches(0.6))

    p=doc.add_paragraph(); p.alignment=WD_ALIGN_PARAGRAPH.CENTER
    r=p.add_run("Table. Geostatistical model selection — "
                "Spatial analysis of PRR in avocado")
    r.bold=True; r.font.size=Pt(13); r.font.color.rgb=rgb("1C2833")
    p2=doc.add_paragraph(); p2.alignment=WD_ALIGN_PARAGRAPH.CENTER
    r2=p2.add_run("Field data — Donmatias-L1 · El Retiro-L2 · La Ceja-L3  |  "
                  "✓ = best model per lot × period (highest R²)")
    r2.italic=True; r2.font.size=Pt(9.5); r2.font.color.rgb=rgb("666666")
    doc.add_paragraph()

    ik_hdrs = ["Lot","Period","Model","p_hat","Nugget","Sill",
               "Range_m","R2","RMSE","CI_pct","✓"]
    uk_hdrs = ["Lot","Period","Model","Nugget","Sill",
               "Range_m","R2","RMSE","CI_pct","✓"]

    add_section(doc, ik_rows, ik_hdrs,
        "Section I — Indicator Kriging (IK): "
        "variogram models for disease incidence (Z = 1 if severity ≥ 1)",
        "1B4F8A")
    add_section(doc, uk_rows, uk_hdrs,
        "Section II — Universal Kriging (UK): "
        "variogram models for disease severity (√AUDPC, linear regional drift)",
        "C0392B")

    note=doc.add_paragraph()
    rn=note.add_run(
        "Note: IK sill bounded to p×(1−p). "
        "UK uses linear regional drift; large lots subsampled to "
        f"{UK_SUBSAMPLE} plants for tractability. "
        "CI (%) = Cambardella index (nugget/sill×100): "
        "<25% strong, 25–75% moderate, >75% weak spatial dependence. "
        "Field data from commercial avocado production lots in Antioquia, Colombia.")
    rn.italic=True; rn.font.size=Pt(8.5); rn.font.color.rgb=rgb("666666")

    out=os.path.join(OUTPUT_DIR,"Table_Geostatistical_Models.docx")
    doc.save(out); print("  Saved: Table_Geostatistical_Models.docx")


# =============================================================================
# 11.  MAIN PIPELINE
# =============================================================================

if __name__ == "__main__":

    # ── 1. Load all lots ───────────────────────────────────────────────────────
    dfs = load_all_lots()
    if not dfs:
        raise RuntimeError(f"No lots loaded. Check '{UNIFIED_XLSX}' and SHEETS.")

    # ── 2. Pre-process ─────────────────────────────────────────────────────────
    audpc_cols: dict = {}
    ind_cols:   dict = {}
    for label, df in dfs.items():
        compute_audpc_sqrt(df)
        _, ic          = compute_indicators(df)
        audpc_cols[label] = [c for c in df.columns
                             if "AUDPC_P" in c and "sqrt" in c]
        ind_cols[label]   = ic

    # ── 3. Variogram analysis ──────────────────────────────────────────────────
    print("\n── IK variogram fitting ──")
    ik_vario: dict = {}; ik_rows: list = []
    for label, df in dfs.items():
        print(f"  {label}:")
        vd, rows         = run_variogram_analysis(df, ind_cols[label], label, is_ik=True)
        ik_vario[label]  = vd; ik_rows += rows

    print("\n── UK variogram fitting ──")
    uk_vario: dict = {}; uk_rows: list = []
    for label, df in dfs.items():
        print(f"  {label}:")
        vd, rows         = run_variogram_analysis(df, audpc_cols[label], label)
        uk_vario[label]  = vd; uk_rows += rows

    # ── 4. Variogram figures ───────────────────────────────────────────────────
    print("\n── Variogram figures ──")
    figure_compact_variograms(
        ik_vario,
        "Indicator variograms of PRR incidence (Z = 1 if severity ≥ 1) — "
        "Three avocado production lots, Antioquia, Colombia",
        "Fig_IK_Variograms")
    figure_compact_variograms(
        uk_vario,
        "Semivariograms for Universal Kriging of PRR severity (√AUDPC) — "
        "Three avocado production lots, Antioquia, Colombia",
        "Fig_UK_Variograms")

    # ── 5. Kriging ─────────────────────────────────────────────────────────────
    print("\n── Indicator Kriging ──")
    ik_kr: dict = {}
    for label, df in dfs.items():
        print(f"  {label}:")
        ik_kr[label] = run_indicator_kriging(df, ik_vario[label], ind_cols[label])

    print("\n── Universal Kriging ──")
    rng    = np.random.default_rng(42)
    uk_kr: dict = {}
    for label, df in dfs.items():
        print(f"  {label}:")
        uk_kr[label] = run_universal_kriging(df, uk_vario[label],
                                              audpc_cols[label], rng)

    # ── 6. IK + UK maps for El Retiro & La Ceja ───────────────────────────────
    ret_lac = [l for l in ["El Retiro-L2","La Ceja-L3"] if l in dfs]
    if len(ret_lac) == 2:
        print("\n── IK + UK maps (El Retiro & La Ceja) ──")
        figure_ik_uk_maps(ik_kr, uk_kr, ret_lac, "Fig_IK_UK_Maps_RetLac")

    # ── 7. Gi* hotspot maps ────────────────────────────────────────────────────
    print("\n── Gi* hotspot maps ──")
    gi_configs = []
    for label in ret_lac:
        df     = dfs[label]
        ranges = [v["params"][2] for v in ik_vario[label]
                  if not v.get("pure_nugget") and "params" in v]
        d_thr  = float(np.median(ranges)) if ranges else 45.0
        print(f"  [{label}]  Gi* threshold: {d_thr:.1f} m  "
              f"(median IK variogram range)")
        gi_configs.append((label, df, d_thr, LOT_CLR[label]))
    if gi_configs:
        figure_gistar(gi_configs, "Fig_GiStar_RetLac")

    # ── 8. Donmatias combined IK + UK figure ───────────────────────────────────
    if "Donmatias-L1" in dfs:
        print("\n── Donmatias combined figure ──")
        figure_donmatias_ik_uk(
            ik_kr["Donmatias-L1"],
            uk_kr["Donmatias-L1"],
            "Fig_Donmatias_IK_UK")

    # ── 9. Geostatistical Word table ───────────────────────────────────────────
    print("\n── Word table ──")
    make_geostats_word_table(ik_rows, uk_rows)

    print("\n✓  All spatial analyses complete.")